# HTMX v4 Minimal Patches

Minimal patches needed for htmx v4 support in FastHTML after alpha7 (raw HTML WebSocket support).

## Setup

In [ ]:
import json
from fastcore.basics import patch
from fastcore.xml import to_xml
from fasthtml.common import *
from fasthtml.core import _find_wsp, _get_htmx, _fix_anno, htmx_exts
from fasthtml.jupyter import *
from functools import partial
from inspect import Parameter
empty = Parameter.empty

## htmx4 Headers

In [ ]:
htmx4src = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha7/dist/htmx.js")
htmx_exts['ws4'] = 'https://unpkg.com/htmx.org@4.0.0-alpha7/dist/ext/hx-ws.js'

def def_hdrs(htmx=True, htmx4=False, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc, scopesrc] + hdrs
    if htmx and htmx4: raise ValueError("Cannot enable both htmx and htmx4")
    if htmx: hdrs = [htmxsrc, fhjsscr] + hdrs
    if htmx4: 
        meta_cfg = Meta(name="htmx:config", content=json.dumps({"metaCharacter": "-"}))
        hdrs = [meta_cfg, htmx4src, fhjsscr] + hdrs 
    return [charset, viewport] + hdrs

## FastHTML.__init__ patch

Maps 'ws' to 'ws4' when htmx4=True, passes htmx4 to def_hdrs.

In [ ]:
from fasthtml.core import _wrap_ex, _list

@patch
def __init__(self: FastHTML, debug=False, routes=None, middleware=None, title="FastHTML page", exception_handlers=None,
             on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
             before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
             secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
             same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
             body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
    middleware, before, after = map(_list, (middleware, before, after))
    self.title, self.canonical, self.session_cookie, self.key_fname = title, canonical, session_cookie, key_fname
    self.htmx4 = htmx4
    hdrs, ftrs, exts = map(listify, (hdrs, ftrs, exts))
    if htmx4 and exts:
        exts = ['ws4' if e in ('ws', 'ws4') else e for e in exts]
    exts = {k: htmx_exts[k] for k in exts}
    htmlkw = htmlkw or {}
    if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
    hdrs += [Script(src=ext) for ext in exts.values()]
    if IN_NOTEBOOK:
        hdrs.append(iframe_scr)
        from IPython.display import display, HTML
        if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
        middleware.append(cors_allow)
    on_startup, on_shutdown = listify(on_startup) or None, listify(on_shutdown) or None
    self.lifespan, self.hdrs, self.ftrs = lifespan, hdrs, ftrs
    self.body_wrap, self.before, self.after, self.htmlkw, self.bodykw = body_wrap, before, after, htmlkw, bodykw
    self.secret_key = get_key(secret_key, key_fname)
    if sess_cls:
        sess = Middleware(sess_cls, secret_key=self.secret_key, session_cookie=session_cookie,
                          max_age=max_age, path=sess_path, same_site=same_site,
                          https_only=sess_https_only, domain=sess_domain)
        middleware.append(sess)
    exception_handlers = ifnone(exception_handlers, {})
    if 404 not in exception_handlers:
        def _not_found(req, exc): return Response('404 Not Found', status_code=404)
        exception_handlers[404] = _not_found
    excs = {k: _wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k, v in exception_handlers.items()}
    super(FastHTML, self).__init__(debug, routes, middleware=middleware, exception_handlers=excs, 
                                    on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)

## _find_wsp patch

htmx v4 sends form data in `data['values']` instead of top-level.

In [ ]:
import fasthtml.core as _core
from fasthtml.core import _send_ws

def _find_wsp_patch(ws, data, hdrs, arg: str, p: Parameter, htmx4=False):
    "Find param named `arg` - checks both top-level (v2) and values dict (v4)"
    anno = p.annotation
    if isinstance(anno, type):
        if issubclass(anno, HtmxHeaders): return _get_htmx(hdrs)
        if issubclass(anno, Starlette): return ws.scope['app']
        if issubclass(anno, WebSocket): return ws
        if issubclass(anno, dict): return data
    if anno is empty:
        if arg.lower() == 'ws': return ws
        if arg.lower() == 'scope': return dict2obj(ws.scope)
        if arg.lower() == 'data': return data
        if arg.lower() == 'htmx': return _get_htmx(hdrs)
        if arg.lower() == 'app': return ws.scope['app']
        if arg.lower() == 'send': return partial(_send_ws, ws)
        if 'session'.startswith(arg.lower()): return ws.scope.get('session', {})
        return None
    res = data.get(arg, None)  # htmx v2: top-level
    if res is empty or res is None: 
        res = data.get('values', {}).get(arg, None)  # htmx v4: in 'values'
    if res is empty or res is None: res = hdrs.get(arg, None)
    if res is empty or res is None: res = p.default
    if not isinstance(res, (list, str)) or anno is empty: return res
    return [_fix_anno(anno, o) for o in res] if isinstance(res, list) else _fix_anno(anno, res)

_core._find_wsp = _find_wsp_patch

## WebSocket Example

Works just like htmx v2 — send raw HTML with `hx_swap_oob='true'`.

Test without hx_swap_oob in ws

In [ ]:
from asyncio import sleep

app = FastHTML(exts='ws', htmx=False, htmx4=True)
rt = app.route

def mk_inp(): return Input(id='msg', name='msg')
nid = 'notifications'

@rt('/')
async def get():
    cts = Div(
        Div(id=nid),
        Form(mk_inp(), id='form', hx_ws_send=True),
        hx_ws_connect='/ws')
    return Titled('Websocket Test', cts)

async def on_connect(send): 
    await send(Div('Hello, you have connected', id=nid, hx_swap_oob=True))

async def on_disconnect(): print('Disconnected!')

@app.ws('/ws', conn=on_connect, disconn=on_disconnect)
async def ws(msg: str, send):
    await send(Div('Hello ' + msg, id=nid, hx_swap_oob=True))
    await sleep(2)
    # Send multiple elements in one message
    await send((
        Div('Goodbye ' + msg, id=nid, hx_swap_oob=True),
        Input(id='msg', name='msg', value='', hx_swap_oob=True)
    ))

srv = JupyUvi(app)